In [ ]:
import os, json
import numpy as np
import pandas as pd
import bop_toolkit_lib as bop
import matplotlib.pyplot as plt

from pathlib import Path

In [ ]:
data_dir = Path("/data/vijay/pose-estimation")
results_dir = data_dir / "results"
filepath = results_dir / "large_lmo_inference/predictions/large-pbrreal-rgb-mmodel_lmo-test_lmo_inference.csv"
targets_json_path = data_dir / "datasets" / "lmo" / "test_targets_bop19.json"
gt_json_path = data_dir / "datasets" / "lmo" / "test/000002/scene_gt.json"

In [ ]:
dataset_name = "ycbv"
data_dir = Path("/data/vijay/pose-estimation")
results_dir = data_dir / "results"
filepath = results_dir / f"large_lmo_inference/predictions/large-pbrreal-rgb-mmodel_{dataset_name}-test_{dataset_name}_inference.csv"
targets_json_path = data_dir / "datasets" / dataset_name / "test_targets_bop19.json"
gt_json_path = data_dir / "datasets" / dataset_name / "test/"

In [ ]:
list(gt_json_path.glob("*/scene_gt.json"))

In [ ]:
def inference_analysis(dataset_name):
    # Paths
    data_dir = Path("/data/vijay/pose-estimation")
    results_dir = data_dir / "results"
    test_dir = data_dir / "datasets" / dataset_name / "test"
    filepath = results_dir / f"large_{dataset_name}_inference/predictions/large-pbrreal-rgb-mmodel_{dataset_name}-test_{dataset_name}_inference.csv"
    targets_json_path = data_dir / "datasets" / dataset_name / "test_targets_bop19.json"
    gt_json_path = list(test_dir.glob("*/scene_gt.json"))


    # Reading files
    gt_labels = {}
    for path in gt_json_path:
        with open(path, "r") as f:
            gt_labels.update(json.loads(f.read()))
    with open(targets_json_path, "r") as f:
        target_labels = json.loads(f.read())
    results = pd.read_csv(filepath).set_index(['scene_id', 'im_id', 'obj_id'])

    # Helper functions
    str2list = lambda x: [*map(float, x.split())]
    list2mat = lambda x: np.array(x).reshape(3, 3)

    def rotation_error(R1, R2):
        trace = np.trace(R1.T @ R2)
        angle = np.arccos((trace - 1) / 2)
        return angle

    # Generating error array
    for i, object in enumerate(target_labels):
        for obj in gt_labels[str(object['im_id'])]:
            if obj['obj_id'] == object['obj_id']:
                object["R_gt"] = obj["cam_R_m2c"]
                break
        if "R_gt" not in object:
            continue
            # raise ValueError
        object["R_pred"] = str2list(results.loc[object['scene_id'], object['im_id'], object['obj_id']].R)
        object["R_template"] = str2list(results.loc[object['scene_id'], object['im_id'], object['obj_id']].R_temp)
        object["R_inplane"] = str2list(results.loc[object['scene_id'], object['im_id'], object['obj_id']].R_inplane)

        r_pred = list2mat(object["R_pred"])
        r_temp = list2mat(object["R_template"])
        r_inpl = list2mat(object["R_inplane"])
        r_gt = list2mat(object["R_gt"])
        r1 = np.linalg.inv(r_inpl) @ r_gt

        object["rotation_error"] = rotation_error(r1, r_temp)

    errors = pd.DataFrame(target_labels).rotation_error

    plt.figure(figsize=(8, 6))
    plt.title(f"Rotation error distribution of selected template in {dataset_name}")
    plt.tight_layout(pad=2.0)
    plt.hist(errors, bins=50)
    os.makedirs("inference_analysis", exist_ok=True)
    plt.savefig(f"inference_analysis/{dataset_name}_inference.png")
    plt.close()

In [ ]:
for dataset_name in ["lmo", "ycbv"]:
    inference_analysis(dataset_name)